# **Import des modules nécessaires**

In [94]:
import pandas as pd #pour la manipulation de données
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte
from spacy.lang.fr.stop_words import STOP_WORDS as spacy_stopwords

from bertopic import BERTopic #pour la modélisation de sujets

from umap import UMAP #pour la réduction de dimensionnalité
from hdbscan import HDBSCAN #pour le clustering de BERTopic
from sklearn.cluster import KMeans #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic

from sentence_transformers import SentenceTransformer #pour les embeddings de phrases

# **Chargement du corpus de Zola et de spacy**


In [69]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",401
1,1865 La confession de Claude.,1865,1,2,Le grillon chantait; le souffle harmonieux des...,336
2,1865 La confession de Claude.,1865,1,3,"Pars cependant, puisque tu as soif de la vie. ...",248
3,1865 La confession de Claude.,1865,1,4,"Ai-je eu trop de confiance en ma force, ma pla...",336
4,1865 La confession de Claude.,1865,1,5,Le premier rayon a chassé le cauchemar de ma v...,359


In [70]:
df.shape

(10638, 6)

# **Traitement du Corpus de Zola**

In [71]:
# Chargement du modèle avec désactivation du 'parser' syntaxique pour gagner en vitesse
# On garde impérativement 'ner' pour repérer les personnages/lieux et 'lemmatizer'
nlp = spacy.load("fr_core_news_lg", disable=["parser"])
nlp.max_length = 2_000_000  

#on convertit en liste
textes_bruts = df["texte"].astype(str).tolist()

textes_nettoyes = []

# Utilisation de nlp.pipe pour traiter les textes par blocs (très rapide)
for doc in nlp.pipe(textes_bruts, batch_size=50): 
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules
        
        if (
            not token.is_stop # Ignorer les stop words spaCy par défaut
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces vides
            and token.ent_type_ not in ['PER', 'LOC', 'ORG'] # Ignorer les Personnages, Lieux et Organisations
            and token.pos_ in {"NOUN", "ADJ", "VERB"}  # Garder Noms, Adjectifs ET Verbes
            and len(lemme) > 2 # Ignorer les mots de 1 ou 2 lettres
        ):
            tokens.append(lemme)
            
    # Rejoindre les tokens validés et les ajouter à la liste finale
    textes_nettoyes.append(" ".join(tokens))

# Application de la liste nettoyée à la nouvelle colonne du DataFrame
df["phrases_lemm"] = textes_nettoyes

# Affichage du résultat
df[["phrases_lemm"]].head()

,phrases_lemm
0,hiver air matin devenir frais mettre manteau b...
1,grillon chanter souffle harmonieux nuit bercer...
2,soif vie projet soi ferme loyal action rêve vi...
3,confiance force place rêver côté jour naître p...
4,rayon chasser cauchemar veille obstacle accept...


## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [72]:
embedding_model = SentenceTransformer(
    "dangvantuan/sentence-camembert-base"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [73]:
print("Génération des embeddings sémantiques...")
embeddings = embedding_model.encode(df['texte'].tolist(), show_progress_bar=True)

Génération des embeddings sémantiques...


Batches:   0%|          | 0/333 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

### 1) HDBSCAN et UMAP

In [117]:
#hdbscan_model = HDBSCAN( min_cluster_size=40, min_samples=2, metric='euclidean', cluster_selection_method='eom',prediction_data=True)

cluster_model = KMeans(n_clusters=30, 
                       random_state=42)

umap_model = UMAP( n_neighbors=20,n_components=8, min_dist=0.0, metric="cosine",random_state=42)

### 2) creation des stop words + vectorisation TF-IDF

In [118]:
stop_perso=[
    "grand", "petit", "homme", "femme", "jour", "heure", "coup", "œil", 
    "main", "bras", "tête", "voix", "milieu", "eau", "terre", "air", "monde", 
    "chose", "nuit", "vie", "enfant", "père", "mère", "fille", "garçon", 
    "monsieur", "madame", "falloir", "aller", "voir", "dire", "faire", 
    "pouvoir", "vouloir", "savoir", "venir", "devoir", "prendre", "donner",
    "oui", "non", "où", "quand", "comment", "deberl", "coupeau", "men", "hein", 
    "tape", "liard", "mercredi"
]

stop_word = list(spacy_stopwords) + stop_perso

### 3) CountVectorizer et ClassTfidfTransformer avec des stop words personnalisés 

In [136]:
vectorizer_model = CountVectorizer(
    stop_words=stop_word,
    min_df=12, #
    max_df=0.40 #   
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
)


topic_model = BERTopic(
    language="french",
    hdbscan_model=cluster_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False,
    verbose=True
)
topics, probs = topic_model.fit_transform(df["phrases_lemm"].tolist(), embeddings= embeddings)

2026-06-04 10:58:05,821 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-04 10:58:15,765 - BERTopic - Dimensionality - Completed ✓
2026-06-04 10:58:15,767 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-04 10:58:15,796 - BERTopic - Cluster - Completed ✓
2026-06-04 10:58:15,799 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-04 10:58:16,640 - BERTopic - Representation - Completed ✓


## **3) Topics Présent**

In [137]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,0,554,0_collectiviste_harmonie_historique_ensemencer,"[collectiviste, harmonie, historique, ensemenc...",[anarchie venir fondre évolution communiste ré...
1,1,546,1_portique_encensoir_oranger_estrade,"[portique, encensoir, oranger, estrade, symbol...",[évoquer savoir splendeur basilique déborder f...
2,2,511,2_phase_impur_tâtonnant_brutaliser,"[phase, impur, tâtonnant, brutaliser, salutair...",[face gonflé oeil doux front limpide disparaît...
3,3,469,3_décret_brusquer_gale_clinique,"[décret, brusquer, gale, clinique, mange, médi...",[intéressant intéressant répéter monsieur reme...
4,4,430,4_flocon_épine_impur_dodo,"[flocon, épine, impur, dodo, réconfort, sépulc...",[demande avorter jeter pierre somme bon action...
5,5,427,5_fagot_tapissier_polisson_sevrer,"[fagot, tapissier, polisson, sevrer, dégrafer,...",[grand table déjeuner bel appétit servir tour ...
6,6,422,6_accusé_expert_paraphe_auditoire,"[accusé, expert, paraphe, auditoire, beaumonta...",[entendre modèle reconnaître venir timbrer par...
7,7,412,7_pudique_insatiable_délai_épine,"[pudique, insatiable, délai, épine, impur, avo...",[fontaine sceller sanctuaire reposer trône cit...
8,8,400,8_prune_saucisse_cochonnerie_soûler,"[prune, saucisse, cochonnerie, soûler, jus, es...",[jour cochon homme poisson quartier mépriser e...
9,9,395,9_crève_cochonnerie_charbonner_décrasser,"[crève, cochonnerie, charbonner, décrasser, fr...",[mourir soir compagnon essayer mâcher feuille ...


In [138]:
# Récupération de la dimension temporelle
timestamps = df['annee'].tolist()

# Génération des topics dans le temps
topics_over_time = topic_model.topics_over_time(
    df['phrases_lemm'].tolist(), 
    timestamps, 
    nr_bins=20 # Optionnel : à ajuster selon le nombre d'années distinctes de ton corpus
)

# Visualisation interactive de l'évolution des 10 premiers topics
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)

19it [00:01, 14.99it/s]


In [ ]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('grand', np.float64(0.017005147472741993)), ('petit', np.float64(0.014208018885504065)), ('milieu', np.float64(0.011649919335881862)), ('heure', np.float64(0.011448751791126012)), ('homme', np.float64(0.011340309172830763)), ('haut', np.float64(0.011135275125658681)), ('coup', np.float64(0.010420100248090423)), ('jour', np.float64(0.010374217810644532)), ('noir', np.float64(0.010220925988676414)), ('terre', np.float64(0.010133559609535362))]

TOPIC 1
[('heure', np.float64(0.01590057452709807)), ('vie', np.float64(0.015721006039242803)), ('femme', np.float64(0.015695122366292842)), ('jour', np.float64(0.015297611576967926)), ('grand', np.float64(0.015239284874121461)), ('amour', np.float64(0.014666138576746177)), ('enfant', np.float64(0.014664660828043958)), ('petit', np.float64(0.013150656817104542)), ('oeil', np.float64(0.013114889067034027)), ('cœur', np.float64(0.01302664749795748))]

TOPIC 2
[('vie', np.float64(0.020569552224391538)), ('cœur', np.float64(0.01979860439510